# multiply-back — ex3: multiply_back across 3-D multi-axis broadcasting

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `multiply-back`. Running the final beacon cell reports progress against the `Backprop: multiply_back` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: multiply_back` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`multiply-back`** (exercise 3). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "multiply-back"
DD_SUBTOPIC = "Backprop: multiply_back"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## multiply_back — 3-D broadcasting and multi-axis unbroadcast

Ex1 covered `(1,4) * (3,4)` (single broadcast axis); ex2 covered Python-float operands. The deepening move stretches both back-fns over a 3-D broadcast where EACH side has a different broadcast axis:

```
x = randn(5, 1, 4)        # singleton dim at axis 1
y = randn(1, 3, 1)        # singleton dims at axes 0 AND 2
out = x * y               # shape (5, 3, 4) — broadcasts both sides
```

Working backward through the per-arg-position rule:

```
grad_x_pre = grad_out * y                          # shape (5,3,4)
grad_x     = unbroadcast(grad_x_pre, x)            # -> (5,1,4), sums axis 1

grad_y_pre = grad_out * x                          # shape (5,3,4)
grad_y     = unbroadcast(grad_y_pre, y)            # -> (1,3,1), sums axes 0 AND 2
```

**Why `unbroadcast` walks multiple axes.** The helper iterates the broadcast rules in reverse: for each axis where `original.shape[i] == 1` but `grad.shape[i] > 1`, sum-reduce with `keepdim=True`. Multi-axis cases exercise the inner loop multiple times — easy to get wrong if the helper uses a single `dim=0` reduction instead of an axis-by-axis sweep.

**Matches `torch.autograd`.** The drill compares the hand-rolled grads to `(x_t * y_t).sum().backward()` results — should match to within 1e-6.

### Exercise 3 — multiply_back across 3-D multi-axis broadcasting

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze multiply_back's behaviour under 3-D broadcasting where each parent has a different singleton-axis pattern, and verify the result matches torch.autograd to within 1e-6 on (5,1,4) * (1,3,1).
> Keywords: multiply-back, 3d, broadcast, multi-axis, unbroadcast
> ```

**KCs targeted:** `unbroadcast-multi-axis-reduce`, `matches-torch-autograd-3d`

Implement `multiply_back0(grad_out, out, x, y)` and `multiply_back1(grad_out, out, x, y)` for the forward op `out = x * y` where `x` and `y` are TENSORS with DIFFERENT broadcasting axes (different singleton dims).

Provided in the stub: `unbroadcast(grad, original)`.

Behaviour:
1. `multiply_back0(grad_out, out, x, y) = unbroadcast(grad_out * y, x)`. The `unbroadcast` reduces axes where `x` was a singleton but the broadcast result was bigger.
2. `multiply_back1(grad_out, out, x, y) = unbroadcast(grad_out * x, y)`. Symmetric.

The test runs the canonical 3-D case `x=(5,1,4)`, `y=(1,3,1)`, `out=(5,3,4)`, and compares your back-fn outputs against `torch.autograd` on the same computation. They must agree to within 1e-6 on every element.

In [ ]:
def unbroadcast(grad, original):
    # Provided helper — sums out axes broadcasting added/expanded.
    while grad.ndim > original.ndim:
        grad = grad.sum(dim=0)
    for i, size in enumerate(original.shape):
        if size == 1 and grad.shape[i] != 1:
            grad = grad.sum(dim=i, keepdim=True)
    return grad


def multiply_back0(grad_out, out, x, y):
    """dL/dx for out = x*y, with multi-axis unbroadcast to x.shape."""
    raise NotImplementedError()


def multiply_back1(grad_out, out, x, y):
    """dL/dy for out = x*y, with multi-axis unbroadcast to y.shape."""
    raise NotImplementedError()


def _test_ex3():
    # === THE HEADLINE: 3-D mixed-singleton broadcast ===
    t.manual_seed(0)
    x = t.randn(5, 1, 4)            # singleton at axis 1
    y = t.randn(1, 3, 1)            # singletons at axes 0 AND 2
    out = x * y                       # shape (5, 3, 4) — both broadcast
    grad_out = t.randn(5, 3, 4)

    g0 = multiply_back0(grad_out, out, x, y)
    g1 = multiply_back1(grad_out, out, x, y)
    assert g0.shape == x.shape, f'g0 shape: got {g0.shape}, expected {x.shape}'
    assert g1.shape == y.shape, f'g1 shape: got {g1.shape}, expected {y.shape}'

    # --- compare with torch.autograd ---
    x_at = x.clone().requires_grad_(True)
    y_at = y.clone().requires_grad_(True)
    loss = (x_at * y_at * grad_out).sum()
    loss.backward()
    assert t.allclose(g0, x_at.grad, atol=1e-6), (
        f'multiply_back0 mismatch with autograd. max |diff| = {(g0 - x_at.grad).abs().max()}'
    )
    assert t.allclose(g1, y_at.grad, atol=1e-6), (
        f'multiply_back1 mismatch with autograd. max |diff| = {(g1 - y_at.grad).abs().max()}'
    )

    # --- expected formula in closed form: g0 sums axis 1 (size 3) of (grad_out * y) ---
    expected_g0 = (grad_out * y).sum(dim=1, keepdim=True)
    assert t.allclose(g0, expected_g0, atol=1e-6), 'g0 != sum over broadcast axis'
    # g1 sums axes 0 (size 5) AND 2 (size 4) of (grad_out * x).
    expected_g1 = (grad_out * x).sum(dim=0, keepdim=True).sum(dim=2, keepdim=True)
    assert t.allclose(g1, expected_g1, atol=1e-6), 'g1 != sum over two broadcast axes'

    # --- 4-D case: (1,1,3,4) * (2,5,1,1) -> out (2,5,3,4); each side has TWO singleton axes ---
    x4 = t.randn(1, 1, 3, 4)
    y4 = t.randn(2, 5, 1, 1)
    out4 = x4 * y4
    go4 = t.randn(2, 5, 3, 4)
    g0_4 = multiply_back0(go4, out4, x4, y4)
    g1_4 = multiply_back1(go4, out4, x4, y4)
    assert g0_4.shape == x4.shape, f'4D g0 shape: {g0_4.shape}'
    assert g1_4.shape == y4.shape, f'4D g1 shape: {g1_4.shape}'
    x_at4 = x4.clone().requires_grad_(True)
    y_at4 = y4.clone().requires_grad_(True)
    (x_at4 * y_at4 * go4).sum().backward()
    assert t.allclose(g0_4, x_at4.grad, atol=1e-6)
    assert t.allclose(g1_4, y_at4.grad, atol=1e-6)

    # --- degenerate case: same-shape (no broadcast) — unbroadcast is identity ---
    x = t.tensor([2.0, 3.0, 4.0])
    y = t.tensor([5.0, 6.0, 7.0])
    out = x * y
    g = t.ones(3)
    g0 = multiply_back0(g, out, x, y)
    g1 = multiply_back1(g, out, x, y)
    assert t.allclose(g0, y) and t.allclose(g1, x), 'same-shape case broke'
    assert g0.shape == x.shape and g1.shape == y.shape

    # --- adversarial: rank-mismatch broadcast (x has fewer dims than y) ---
    x_lr = t.tensor([2.0, 3.0])     # shape (2,)
    y_lr = t.tensor([[1.0, 1.0],    # shape (3, 2)
                     [2.0, 2.0],
                     [3.0, 3.0]])
    out_lr = x_lr * y_lr   # (3, 2) via implicit leading 1
    go_lr = t.ones(3, 2)
    g0_lr = multiply_back0(go_lr, out_lr, x_lr, y_lr)
    g1_lr = multiply_back1(go_lr, out_lr, x_lr, y_lr)
    assert g0_lr.shape == x_lr.shape, f'lower-rank g0 shape: {g0_lr.shape}'
    assert g1_lr.shape == y_lr.shape, f'lower-rank g1 shape: {g1_lr.shape}'
    x_at = x_lr.clone().requires_grad_(True)
    y_at = y_lr.clone().requires_grad_(True)
    (x_at * y_at * go_lr).sum().backward()
    assert t.allclose(g0_lr, x_at.grad, atol=1e-6)
    assert t.allclose(g1_lr, y_at.grad, atol=1e-6)
    _dd_passed.add('ex3')
    print("ex3 ✓")

_test_ex3()

<details><summary>Solution</summary>

```python
def multiply_back0(grad_out, out, x, y):
    return unbroadcast(grad_out * y, x)


def multiply_back1(grad_out, out, x, y):
    return unbroadcast(grad_out * x, y)
```

**The unbroadcast helper is doing all the work.** Both back-fns are one-liners; the multi-axis reduction is the part that gets hard. The helper's inner loop is what handles `(1,3,1)` correctly: the rank-equal check is satisfied at the start (both grad and original are 3-D after the multiplication), then the per-axis loop sums axes 0 and 2 with `keepdim=True`.

**Why `keepdim=True`.** Without it, summing axis 0 of `(5,3,4)` would give `(3,4)` and the next axis index would refer to a different axis. `keepdim=True` keeps the rank constant so the loop indices stay valid.

**The rank-mismatch test exercises the OUTER `while` loop.** When `x_lr` has shape `(2,)` (rank 1) but `grad_out * y_lr` has shape `(3,2)` (rank 2), the `while grad.ndim > original.ndim` guard fires once and sums out the leading axis. The inner loop then finds no singleton-to-broadcast axes in `(2,)` and exits. Two-stage reduction — both stages must work.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()